# **Preparation Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
# <Student to fill this section and then remove this comment>
group_name = "Neural Minds"
student_name = "Aryaman Malik"
student_id = "25980501"

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
!pip install -q scikit-learn
!pip install -q "vegafusion[embed]>=1.5.0"
!pip install -q "vl-convert-python>=1.6.0"


### 0.b Import Packages

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Enable vegafusion for Altair large dataset support
alt.data_transformers.enable("vegafusion")

---
## A. Feature Selection


## A.0 Load Data

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load datasets
try:
  customer_df = pd.read_csv(at.folder_path / "customer.csv")
  person_df = pd.read_csv(at.folder_path / "person.csv")
  product_category_df = pd.read_csv(at.folder_path / "product_category.csv")
  product_cost_history_df = pd.read_csv(at.folder_path / "product_cost_history.csv")
  product_list_price_history_df = pd.read_csv(at.folder_path / "product_list_price_history.csv")
  product_sub_category_df = pd.read_csv(at.folder_path / "product_sub_category.csv")
  product_df = pd.read_csv(at.folder_path / "product.csv")
  sales_order_detail_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
  sales_order_header_df = pd.read_csv(at.folder_path / "sales_order_header.csv")
  sales_territory_df = pd.read_csv(at.folder_path / "sales_territory.csv")
  special_offer_product_df = pd.read_csv(at.folder_path / "special_offer_product.csv")
  special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")
  store_df = pd.read_csv(at.folder_path / "store.csv")
  unit_measure_df = pd.read_csv(at.folder_path / "unit_measure.csv")
except Exception as e:
  print(e)

### A.1 Approach 1

In [ ]:
print("sales_order_detail columns:", sales_order_detail_df.columns.tolist())
print(f"sales_order_detail shape: {sales_order_detail_df.shape}")
print("\nsales_order_header columns:", sales_order_header_df.columns.tolist())
print(f"sales_order_header shape: {sales_order_header_df.shape}")

In [ ]:
domain_features = {
    "Line transaction": ["order_quantity", "unit_price", "unit_price_discount", "line_total"],
    "Product pricing (joined)": ["list_price", "standard_cost"],
    "Order context (joined from header)": ["online_order_flag", "order_date", "territory_id"],
    "Product hierarchy (joined)": ["category_name", "subcategory_name", "class"],
    "Promotion context (joined from offer)": ["offered_discount_pct", "offer_type", "offer_category"]
}
for bucket, cols in domain_features.items():
    print(f"\n{bucket}:")
    for c in cols:
        print(f"  - {c}")

In [ ]:
feature_selection_1_insights = """
Approach 1 is a domain-driven shortlist built around the question 'what does a normal sales line look like?'. A normal line has four kinds of context, and an anomaly is a violation in one or more of them:

1. Line transaction (order_quantity, unit_price, unit_price_discount, line_total): the raw facts of the transaction.
2. Product pricing reference (list_price, standard_cost): the company's published price and internal cost - these anchor what unit_price SHOULD be.
3. Order context (online_order_flag, order_date, territory_id): online retail orders behave very differently from offline reseller orders.
4. Product hierarchy (category_name, subcategory_name): a USD 50 line is normal for a Saddle, impossible for a Road Bike. Conditioning on hierarchy is essential.
5. Promotion context (offered_discount_pct, offer_type): if a line has a discount and a matching active special offer, it is sanctioned, not anomalous.

These 14 raw columns plus engineered ratios will form the candidate feature set. They are picked because every one of them is plausibly used by a human auditor reviewing a flagged transaction.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2

In [ ]:
df_check = (sales_order_detail_df
            .merge(product_df.drop_duplicates('product_id'),
                   on='product_id', how='left', suffixes=('','_prod'))
            .merge(sales_order_header_df[['sales_order_id','order_date','online_order_flag','territory_id']],
                   on='sales_order_id', how='left'))
df_check = df_check[df_check['list_price'] > 0]
print(f"Joined dataset: {len(df_check)} rows, {len(df_check.columns)} columns")

miss = df_check.isnull().mean().sort_values(ascending=False) * 100
print("\nTop 12 columns by missingness (%):")
print(miss.head(12).round(2))

numeric = df_check.select_dtypes(include=['float64','int64'])
zero_var = numeric.columns[numeric.std() == 0].tolist()
print(f"\nNumeric columns with std == 0 (constant): {zero_var}")

In [ ]:
drop_for_sparsity = [
    'discontinued_date','sell_end_date','sell_start_date',
    'size','size_unit_measure_code','weight','weight_unit_measure_code',
    'style','color','product_line','product_model_id',
    'product_number','name','is_sellable'
]
keep_after_filter = [c for c in df_check.columns if c not in drop_for_sparsity]
print(f"After variance/missingness filter: kept {len(keep_after_filter)} of {len(df_check.columns)} columns")
print(f"Dropped: {[c for c in drop_for_sparsity if c in df_check.columns]}")

In [ ]:
# <Student to fill this section>
feature_selection_2_insights = """
Approach 2 applies a missingness and variance filter to the 33-column joined dataset:
- discontinued_date is 100% null - dropped.
- sell_end_date is 92% null - the few non-null values represent recently retired products only, not useful as a feature.
- size, size_unit_measure_code, weight, weight_unit_measure_code, style, color, product_line, product_model_id are all >35% null. They describe physical properties that exist for bikes/clothing but not for components/accessories. Imputing them at this scale would introduce more noise than signal.
- name (product name) is high-cardinality free text with ~20% missing.
- product_number is a high-cardinality SKU code with ~10% missing.
- is_sellable has zero variance (all True for priced products) - drops out automatically.

This narrows the candidate set from 33 columns to 19. We will further trim bookkeeping IDs (sales_order_id, sales_order_detail_id, product_id, territory_id, customer_id, special_offer_id, product_subcategory_id, product_category_id) and the sparse class column when we build the final model feature set in Section E, leaving a clean ~10 raw columns that we will enrich with engineered ratios.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.3 Approach 3 "\<Channel stratified inspection\>"

> You can add more cells related to other approaches in this section

In [ ]:
tmp = df_check.copy()
tmp['price_vs_listprice'] = tmp['unit_price'] / tmp['list_price']

channel_summary = tmp.groupby('online_order_flag').agg(
    n_lines=('unit_price','size'),
    mean_ratio=('price_vs_listprice','mean'),
    median_ratio=('price_vs_listprice','median'),
    min_ratio=('price_vs_listprice','min'),
    max_ratio=('price_vs_listprice','max'),
    mean_qty=('order_quantity','mean'),
    median_qty=('order_quantity','median'),
    max_qty=('order_quantity','max')
).round(3)
print("Behaviour by channel (online_order_flag):")
print(channel_summary)

In [ ]:
offline = tmp[tmp['online_order_flag']==0.0]
print(f"Offline lines at exactly 60% of list_price: "
      f"{((offline['price_vs_listprice'] - 0.60).abs() < 0.001).sum()} / {len(offline)} "
      f"({((offline['price_vs_listprice'] - 0.60).abs() < 0.001).mean()*100:.1f}%)")
print(f"Offline median price ratio: {offline['price_vs_listprice'].median():.3f}")
print(f"Online median price ratio:  {tmp[tmp['online_order_flag']==1.0]['price_vs_listprice'].median():.3f}")

In [ ]:
# <Student to fill this section>
feature_selection_n_insights = """
Approach 3 confirms that online_order_flag is not just another feature - it is a structural stratifier of pricing behaviour:
- Online lines (31,434, 74.7% of transactions): unit_price equals list_price for the median line, with the entire ratio tightly bounded between 0.893 and 1.000. Every online line has order_quantity = 1.
- Offline lines (10,666, 25.3%): unit_price is tightly clustered at 60% of list_price - 8,776 (82.3%) sit at exactly this rate. Order quantities are larger (median 3, max 36).

Two consequences for feature selection and labelling:
1. online_order_flag is a mandatory feature - the model must know which regime each line belongs to.
2. Any anomaly rule based on the price/list ratio must be channel-conditional. A 60% ratio is anomalous online but the default offline. We will encode this directly in the pseudo-labels in Section C.

This approach also justifies adding one extra engineered feature - prod_offline_median_ratio, the median price ratio for THIS product in the offline channel - so we can detect 'this line was discounted further than this product is usually discounted to resellers', the real offline anomaly pattern.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### A.z Final Selection of Features

In [ ]:
features_list = [
    'order_quantity', 'unit_price', 'unit_price_discount', 'line_total',
    'list_price', 'standard_cost',
    'online_order_flag', 'order_date', 'territory_id',
    'category_name', 'subcategory_name',
    'offered_discount_pct', 'offer_type', 'offer_category'
]
print(f"Selected raw features ({len(features_list)}):")
for f in features_list:
    print(f"  - {f}")

In [ ]:
engineered_features_preview = [
    "net_unit_price             = unit_price * (1 - unit_price_discount)",
    "price_vs_listprice         = unit_price / list_price",
    "margin_pct                 = (net_unit_price - standard_cost) / net_unit_price * 100",
    "price_vs_subcat_median     = net_unit_price / subcategory_median(net_unit_price)",
    "qty_vs_subcat_median       = order_quantity / subcategory_median(order_quantity)",
    "prod_offline_median_ratio  = median offline price-vs-list ratio for this product",
    "line_total_computed        = order_quantity * net_unit_price",
    "order_year, order_month, order_dow = decomposed from order_date"
]
for f in engineered_features_preview:
    print(f"  + {f}")

In [ ]:
feature_selection_explanations = """
The final raw feature set has 14 columns sourced from 6 tables: sales_order_detail, sales_order_header, product, product_sub_category, product_category, and special_offer + special_offer_product.

Three filtering principles were applied:
1. Domain relevance (Approach 1): every kept feature is one a human auditor would consult when reviewing a flagged sale.
2. Information content (Approach 2): columns with >35% missingness, no variance, or pure-text high-cardinality content were dropped (size, weight, color, style, product name, etc.).
3. Channel awareness (Approach 3): online_order_flag is treated as a structural stratifier, not just another categorical.

In Section D we will engineer 8 additional derived features (net price, price/list ratio, margin, subcategory-relative ratios, channel-conditional product baseline, line-total sanity, and date decompositions). The model will see a final feature set of 27 columns after one-hot encoding the categoricals.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

### B.1 Fixing "\<Duplicate product_id rows\>"

In [ ]:
print(f"product_df before dedup: {len(product_df)} rows, {product_df['product_id'].nunique()} unique product_id")
product_clean = product_df.drop_duplicates(subset='product_id').reset_index(drop=True)
print(f"product_clean after dedup: {len(product_clean)} rows ({len(product_df) - len(product_clean)} duplicates removed)")

In [ ]:
sparse_cols = ['discontinued_date','size','size_unit_measure_code','weight',
               'weight_unit_measure_code','style','color','product_line','product_model_id',
               'sell_start_date','sell_end_date','product_number','name','is_sellable',
               'is_manufactured','safety_stock_level','reorder_point','days_to_manufacture']
product_clean = product_clean.drop(columns=[c for c in sparse_cols if c in product_clean.columns])
print(f"After dropping sparse cols: {product_clean.shape}")
print(f"Remaining columns: {product_clean.columns.tolist()}")

product_clean = product_clean[product_clean['list_price'] > 0].reset_index(drop=True)
print(f"\nAfter filter list_price > 0: {len(product_clean)} priced products")

n_null = product_clean['standard_cost'].isnull().sum()
print(f"\nstandard_cost nulls before impute: {n_null}")
sub_cost_med = product_clean.groupby('product_subcategory_id')['standard_cost'].transform('median')
product_clean['standard_cost'] = product_clean['standard_cost'].fillna(sub_cost_med)
print(f"standard_cost nulls after subcategory-median impute: {product_clean['standard_cost'].isnull().sum()}")

In [ ]:
data_cleaning_1_explanations = """
Fix 1: Deduplicate product.csv.
The product master table contained 886 rows but only 504 unique product_id values. Inspection in the EDA notebook confirmed that the 382 duplicate rows are fully identical (every column matches), so the duplication is a load/export artefact and the correct fix is drop_duplicates(subset='product_id'). Without this fix every downstream join would multiply each sales line 2-6x and corrupt every aggregate.

We also drop 18 sparse / uninformative columns from product (discontinued_date, size, weight, style, color, product_line, product_model_id, etc., plus zero-variance is_sellable and bookkeeping fields like safety_stock_level), filter to priced products only (list_price > 0, drops 200 raw-material rows), and impute the 32 remaining null standard_cost values using the median cost within each product subcategory. After imputation only 2 nulls remain (in subcategories where every product has a null cost); these will be handled at category level in B.3.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing "\<Build_the_joined_line-item_dataset\>"

In [ ]:
prod_hier = (product_clean
             .merge(product_sub_category_df.rename(columns={'name':'subcategory_name'}),
                    on='product_subcategory_id', how='left')
             .merge(product_category_df.rename(columns={'name':'category_name'}),
                    on='product_category_id', how='left'))
print(f"Product hierarchy shape: {prod_hier.shape}")
print(f"Columns: {prod_hier.columns.tolist()}")

In [ ]:
m1 = sales_order_detail_df.merge(prod_hier, on='product_id', how='inner')
print(f"After product join (inner): {m1.shape}  (dropped {len(sales_order_detail_df)-len(m1)} raw-material lines)")

m2 = m1.merge(sales_order_header_df[['sales_order_id','order_date','online_order_flag','territory_id','customer_id']],
              on='sales_order_id', how='left')
print(f"After header join: {m2.shape}, order_date nulls: {m2['order_date'].isnull().sum()}")

so_join = special_offer_product_df.merge(
    special_offer_df[['special_offer_id','discount_pct','type','category']], on='special_offer_id', how='left')
df_clean = m2.merge(so_join.rename(columns={'discount_pct':'offered_discount_pct',
                                              'type':'offer_type',
                                              'category':'offer_category'}),
                    on=['special_offer_id','product_id'], how='left')
print(f"After special offer join: {df_clean.shape}, no-offer lines: {df_clean['offered_discount_pct'].isnull().sum()}")
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
print(f"Final cleaned & joined: {df_clean.shape}")
print(f"Date range: {df_clean['order_date'].min().date()} to {df_clean['order_date'].max().date()}")

In [ ]:
data_cleaning_2_explanations = """
Fix 2: Build the joined line-item dataset.
The unit of analysis for anomaly detection is a single sales-order line item. We join 6 tables to get a complete picture of each line:

- sales_order_detail (42,100 lines): the transactional core - quantity, price, discount, line_total, product_id, special_offer_id.
- product (deduplicated, priced only): standard_cost, list_price, subcategory and category.
- product_sub_category + product_category: human-readable hierarchy names.
- sales_order_header: order_date, online_order_flag (channel), territory_id, customer_id.
- special_offer + special_offer_product: the discount_pct that was OFFERED for this combination (independent of what was actually applied).

The product join is INNER because we only want lines for priced products. No lines drop here in practice because zero-priced raw materials never appear in sales transactions. The header and offer joins are both effectively complete (every line has a parent order and a matching offer, including the No Discount default).

Final cleaned dataset: 42,100 rows x 22 columns spanning 30 May 2011 to 29 June 2014.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "\<Referential-integrity_check_and_final_cost_imputation\>"

In [ ]:
key_cols = ['order_quantity','unit_price','unit_price_discount','line_total','list_price',
            'standard_cost','online_order_flag','category_name','subcategory_name',
            'offered_discount_pct','offer_type']
print("Null counts in key columns:")
print(df_clean[key_cols].isnull().sum())

df_clean['_check_total'] = df_clean['order_quantity'] * df_clean['unit_price'] * (1 - df_clean['unit_price_discount'])
diff = (df_clean['line_total'] - df_clean['_check_total']).abs()
print(f"\nLines with line_total discrepancy > 0.01: {(diff > 0.01).sum()}")
print(f"Max discrepancy: {diff.max():.6f}")
df_clean = df_clean.drop(columns=['_check_total'])

print(f"\nOnline / offline split:")
print(df_clean['online_order_flag'].value_counts(dropna=False))


In [ ]:
print(f"standard_cost nulls in joined data: {df_clean['standard_cost'].isnull().sum()}")
cat_cost_med = df_clean.groupby('category_name')['standard_cost'].transform('median')
df_clean['standard_cost'] = df_clean['standard_cost'].fillna(cat_cost_med)
print(f"After category-median impute: {df_clean['standard_cost'].isnull().sum()}")

In [ ]:
data_cleaning_n_explanations = """
Fix 3: Validate referential integrity and resolve remaining nulls.

The joined dataset is structurally clean - all key columns are populated except standard_cost (732 nulls, ~1.7%). These correspond to the 2 products whose subcategory had no costed peers (carried over from B.1). We impute these with the median standard_cost within their parent CATEGORY, which leaves zero nulls.

Sanity check on line_total: the dataset's line_total column should equal order_quantity * unit_price * (1 - unit_price_discount). The check confirms this holds across all 42,100 lines with zero discrepancy. We therefore will NOT use line_total minus computed_line_total as an anomaly signal because there is none. Instead, the line_total_computed feature engineered in Section D acts as a redundant magnitude input the model can split on.

Channel split confirmed: 31,434 online (74.7%) vs 10,666 offline (25.3%), matching the Approach 3 observation.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_n_explanations)

### B.n Fixing "\<Final_Cleaning_Summary\>"

> You can add more cells related to other issues in this section

In [ ]:
print(f"Final cleaned dataset shape: {df_clean.shape}")
print(f"\nColumns ({len(df_clean.columns)}):")
for c in df_clean.columns:
    print(f"  {c:<30} dtype={df_clean[c].dtype}  nulls={df_clean[c].isnull().sum()}")

In [ ]:
print(f"Memory: {df_clean.memory_usage(deep=True).sum()/1024**2:.2f} MB")
print("\nUnique values per categorical:")
for c in ['online_order_flag','category_name','subcategory_name','offer_type','offer_category','territory_id','class']:
    if c in df_clean.columns:
        print(f"  {c}: {df_clean[c].nunique()} unique")

In [ ]:
data_cleaning_n_explanations = """
Cleaning is complete. The dataset is now 42,100 rows x 22 columns with:
- Zero nulls in any column used by the model.
- Zero referential-integrity issues.
- Verified line_total internal consistency (max discrepancy 0.0).
- Categorical cardinality is manageable: 2 channels, 4 categories, 35 subcategories used, 6 offer types, 4 territories.

Key transformations:
1. Deduplicated product.csv (886 -> 504 rows).
2. Dropped 18 sparse/uninformative columns from product.
3. Filtered to priced products (list_price > 0), reducing master from 504 to 304.
4. Inner-joined the product hierarchy to sales_order_detail.
5. Left-joined header context and special-offer context.
6. Imputed standard_cost nulls hierarchically (subcategory median -> category median).
7. Verified line_total = quantity * unit_price * (1 - discount) holds for all rows.

The class column has 54% nulls and will be dropped from the model feature set in Section E.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_n_explanations', value=data_cleaning_n_explanations)

---
## C. Split Datasets


In [ ]:
# Step 1: derive helper columns the labels depend on
df_clean['net_unit_price']    = df_clean['unit_price'] * (1 - df_clean['unit_price_discount'])
df_clean['price_vs_listprice'] = df_clean['unit_price'] / df_clean['list_price']

# Step 2: product-specific offline baseline
offline = df_clean[df_clean['online_order_flag'] == 0.0]
prod_offline_median = (offline.groupby('product_id')['price_vs_listprice']
                              .median()
                              .rename('prod_offline_median_ratio'))
df_clean = df_clean.merge(prod_offline_median, on='product_id', how='left')

# Step 3: subcategory 99th-percentile of order_quantity
sub_q99 = (df_clean.groupby('subcategory_name')['order_quantity']
                   .quantile(0.99)
                   .rename('subcat_q99_qty'))
df_clean = df_clean.merge(sub_q99, on='subcategory_name', how='left')

# Step 4: the three pseudo-labels
df_clean['label_unusual_online_discount'] = (
    (df_clean['online_order_flag'] == 1.0) & (df_clean['price_vs_listprice'] < 0.95)
).astype(int)
df_clean['label_unusual_offline_discount'] = (
    (df_clean['online_order_flag'] == 0.0)
    & df_clean['prod_offline_median_ratio'].notna()
    & ((df_clean['prod_offline_median_ratio'] - df_clean['price_vs_listprice']) > 0.10)
).astype(int)
df_clean['label_extreme_qty'] = (
    (df_clean['order_quantity'] > df_clean['subcat_q99_qty']) & (df_clean['order_quantity'] >= 5)
).astype(int)
df_clean['is_anomaly'] = df_clean[
    ['label_unusual_online_discount','label_unusual_offline_discount','label_extreme_qty']
].max(axis=1).astype(int)

print("Label breakdown on full dataset (n=42,100):")
print(f"  unusual_online_discount:  {df_clean['label_unusual_online_discount'].sum():5d} ({df_clean['label_unusual_online_discount'].mean()*100:.2f}%)")
print(f"  unusual_offline_discount: {df_clean['label_unusual_offline_discount'].sum():5d} ({df_clean['label_unusual_offline_discount'].mean()*100:.2f}%)")
print(f"  extreme_qty:              {df_clean['label_extreme_qty'].sum():5d} ({df_clean['label_extreme_qty'].mean()*100:.2f}%)")
print(f"  ANY anomaly:              {df_clean['is_anomaly'].sum():5d} ({df_clean['is_anomaly'].mean()*100:.2f}%)")

# Step 5: stratified 60 / 20 / 20 split
training_df, temp_df = train_test_split(df_clean, test_size=0.40,
                                         stratify=df_clean['is_anomaly'], random_state=42)
validation_df, testing_df = train_test_split(temp_df, test_size=0.50,
                                              stratify=temp_df['is_anomaly'], random_state=42)
training_df   = training_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
testing_df    = testing_df.reset_index(drop=True)

print(f"\nSplit sizes:")
print(f"  training_df:   {len(training_df):6d}  anomalies={training_df['is_anomaly'].sum():4d}  rate={training_df['is_anomaly'].mean()*100:.2f}%")
print(f"  validation_df: {len(validation_df):6d}  anomalies={validation_df['is_anomaly'].sum():4d}  rate={validation_df['is_anomaly'].mean()*100:.2f}%")
print(f"  testing_df:    {len(testing_df):6d}  anomalies={testing_df['is_anomaly'].sum():4d}  rate={testing_df['is_anomaly'].mean()*100:.2f}%")

In [ ]:
import altair as alt
breakdown = pd.DataFrame([
    {'split':'train', 'label':'online_disc',  'count': training_df['label_unusual_online_discount'].sum()},
    {'split':'train', 'label':'offline_disc', 'count': training_df['label_unusual_offline_discount'].sum()},
    {'split':'train', 'label':'extreme_qty',  'count': training_df['label_extreme_qty'].sum()},
    {'split':'val',   'label':'online_disc',  'count': validation_df['label_unusual_online_discount'].sum()},
    {'split':'val',   'label':'offline_disc', 'count': validation_df['label_unusual_offline_discount'].sum()},
    {'split':'val',   'label':'extreme_qty',  'count': validation_df['label_extreme_qty'].sum()},
    {'split':'test',  'label':'online_disc',  'count': testing_df['label_unusual_online_discount'].sum()},
    {'split':'test',  'label':'offline_disc', 'count': testing_df['label_unusual_offline_discount'].sum()},
    {'split':'test',  'label':'extreme_qty',  'count': testing_df['label_extreme_qty'].sum()},
])
chart = alt.Chart(breakdown).mark_bar().encode(
    x=alt.X('split:N', sort=['train','val','test'], title='Split'),
    y=alt.Y('count:Q', title='Number of anomalies'),
    color=alt.Color('label:N', title='Anomaly type'),
    xOffset='label:N',
    tooltip=['split','label','count']
).properties(title='Anomaly composition per split', width=400, height=300)
chart.display()

In [ ]:
# <Student to fill this section>
data_splitting_explanations = """
Splitting strategy: 60 / 20 / 20 stratified random split by is_anomaly, random_state=42.

Why stratified random (not chronological):
We considered two options and chose stratified random with explicit reasoning.

1. The natural choice for an anomaly-detection problem is a CHRONOLOGICAL split (train on the oldest 60% of orders, validate on the next 20%, test on the most recent 20%). This best matches the production setting where the model would be deployed to score new orders as they arrive.

2. However, in this dataset the time distribution of anomalies is highly uneven. The 'unusual online discount' lines concentrate in 2011-2013 and drop sharply by 2014. A chronological split produced ~9% anomalies in training but only 0.2% in validation and 0.9% in test (18 and 72 anomalies respectively). That is too few to compute trustworthy precision / recall, especially per-anomaly-type.

3. The 60 / 20 / 20 STRATIFIED random split guarantees ~5.58% anomalies in each partition (1,411 / 470 / 470), enabling reliable evaluation and breakdowns by anomaly type. The trade-off is that we lose the explicit time-aware validation story.

To preserve some temporal signal we (a) keep order_year, order_month and order_dow as features, and (b) recommend in the final report that production deployment validate the model on a recent unseen month before going live.

Stratification variable: is_anomaly (the combined label). This keeps the overall rate constant across splits.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

---
## D. Feature Engineering

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df.copy()
  validation_df_eng = validation_df.copy()
  testing_df_eng = testing_df.copy()
except Exception as e:
  print(e)

### D.1 New Feature: <net_unit_price, price_vs_listprice, margin_pct>



In [ ]:
def add_price_features(d):
    d['net_unit_price']     = d['unit_price'] * (1 - d['unit_price_discount'])
    d['price_vs_listprice'] = d['unit_price'] / d['list_price']
    d['margin_pct']         = np.where(d['standard_cost'] > 0,
                                       (d['net_unit_price'] - d['standard_cost']) / d['net_unit_price'] * 100,
                                       np.nan)
    return d

training_df_eng   = add_price_features(training_df_eng)
validation_df_eng = add_price_features(validation_df_eng)
testing_df_eng    = add_price_features(testing_df_eng)

print("D.1 added: net_unit_price, price_vs_listprice, margin_pct")
print("\nSummary on training set:")
print(training_df_eng[['net_unit_price','price_vs_listprice','margin_pct']].describe().round(3))

In [ ]:
hist = alt.Chart(training_df_eng).mark_bar(opacity=0.7).encode(
    x=alt.X('price_vs_listprice:Q', bin=alt.Bin(maxbins=30), title='unit_price / list_price'),
    y=alt.Y('count():Q', title='Number of lines'),
    color=alt.Color('online_order_flag:N', title='Channel')
).properties(title='price_vs_listprice distribution by channel (train)',
             width=600, height=300)
hist.display()

In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_1_explanations = """
D.1 adds three price-and-margin features:

1. net_unit_price = unit_price * (1 - unit_price_discount): actual revenue per unit after any applied discount. Range USD 1.30 to USD 3,578 on the training set.
2. price_vs_listprice = unit_price / list_price: a scale-invariant ratio comparing what the customer paid to the published price. Median 1.000 (online dominates); ranges from 0.20 to 1.00.
3. margin_pct = (net_unit_price - standard_cost) / net_unit_price * 100: gross margin percent realised on this line. Median 45.4% but min -354% on training - some offline reseller lines DO sell below cost (the wholesale rate is set as a fixed 60% of list price regardless of margin). These will appear naturally in the data; the model needs to distinguish 'expected below-cost wholesale' from 'unauthorised deep discount'.

These features are the core 'is this line priced normally?' indicators. They are deterministic functions of existing columns so they introduce no label leakage.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### D.2 New Feature "\<subcategory_relative_ratios\>"



In [ ]:
sub_med_qty   = training_df_eng.groupby('subcategory_name')['order_quantity'].median().rename('subcat_median_qty')
sub_med_price = training_df_eng.groupby('subcategory_name')['net_unit_price'].median().rename('subcat_median_price')

def add_subcat_ratios(d, mq, mp):
    d = d.merge(mq, on='subcategory_name', how='left')
    d = d.merge(mp, on='subcategory_name', how='left')
    d['qty_vs_subcat_median']    = d['order_quantity'] / d['subcat_median_qty']
    d['price_vs_subcat_median']  = d['net_unit_price'] / d['subcat_median_price']
    return d

training_df_eng   = add_subcat_ratios(training_df_eng, sub_med_qty, sub_med_price)
validation_df_eng = add_subcat_ratios(validation_df_eng, sub_med_qty, sub_med_price)
testing_df_eng    = add_subcat_ratios(testing_df_eng, sub_med_qty, sub_med_price)

print("D.2 added: qty_vs_subcat_median, price_vs_subcat_median (and helper columns)")
print(f"\ntrain shape: {training_df_eng.shape}")
print("\nSummary on training set:")
print(training_df_eng[['qty_vs_subcat_median','price_vs_subcat_median']].describe().round(3))

In [ ]:
chart = alt.Chart(training_df_eng).mark_point(opacity=0.3).encode(
    x=alt.X('order_quantity:Q', scale=alt.Scale(type='log'), title='order_quantity (log)'),
    y=alt.Y('qty_vs_subcat_median:Q', scale=alt.Scale(type='log'), title='qty / subcat median (log)'),
    color=alt.Color('label_extreme_qty:N', title='extreme_qty label',
                    scale=alt.Scale(domain=[0,1], range=['#888','#E45756'])),
    tooltip=['subcategory_name','order_quantity','qty_vs_subcat_median']
).properties(title='Quantity outliers light up at the top right', width=550, height=350)
chart.display()

In [ ]:
feature_engineering_2_explanations = """
D.2 adds two subcategory-relative ratios:

1. qty_vs_subcat_median: how many times the typical order quantity for this subcategory was bought on this line. Median 1.00 (most lines buy the typical quantity), but the tail extends up to 27x in training. Highly correlated with the extreme_qty label.
2. price_vs_subcat_median: how the line's effective price compares to the median for its subcategory. Median 1.00; the tail in both directions captures relative outliers (lines priced 3-7x the subcategory median or down at 0.03x).

CRUCIAL: the subcategory medians are computed on the TRAINING set only and then merged onto val/test using the train-derived mapping. This prevents data leakage - val/test medians never inform features used during training.

Together with the D.1 features these provide both ABSOLUTE pricing signals (price_vs_listprice, margin_pct) and RELATIVE/RANK signals (price_vs_subcat_median, qty_vs_subcat_median). Tree-based anomaly detectors like Isolation Forest exploit rank structure, so both views are useful.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### D.3 New Feature "\<Temporal Decomposition\>"



In [ ]:
# <Student to fill this section and then remove this comment>
def add_date_features(d):
    d['order_date']  = pd.to_datetime(d['order_date'])
    d['order_year']  = d['order_date'].dt.year
    d['order_month'] = d['order_date'].dt.month
    d['order_dow']   = d['order_date'].dt.dayofweek
    return d

training_df_eng   = add_date_features(training_df_eng)
validation_df_eng = add_date_features(validation_df_eng)
testing_df_eng    = add_date_features(testing_df_eng)

print("D.3 added: order_year, order_month, order_dow")
print("\nDistribution on training set:")
print("Years:", dict(training_df_eng['order_year'].value_counts().sort_index()))
print("Day-of-week:", dict(training_df_eng['order_dow'].value_counts().sort_index()))

In [ ]:
yr = (training_df_eng.groupby(['order_year','is_anomaly']).size().reset_index(name='count'))
chart = alt.Chart(yr).mark_bar().encode(
    x=alt.X('order_year:O', title='Order year'),
    y=alt.Y('count:Q', title='Number of lines'),
    color=alt.Color('is_anomaly:N', title='Anomaly',
                    scale=alt.Scale(domain=[0,1], range=['#4C78A8','#E45756']))
).properties(title='Lines per year on training set, anomalies highlighted',
             width=400, height=300)
chart.display()

In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_3_explanations = """
D.3 adds three temporal features by decomposing order_date:

1. order_year (2011-2014): captures multi-year trends. Training data is heavily weighted to 2013-2014 (22,842 of 25,260 lines).
2. order_month (1-12): captures seasonality - end-of-year clearance, mid-year sales, etc.
3. order_dow (0-6, Mon-Sun): captures weekday/weekend patterns. Distribution is fairly uniform across weekdays with a small Sunday peak.

Why include these even though we used a stratified random split? Because real anomaly patterns may correlate with calendar events (flash sales not recorded as special offers, end-of-quarter clearance, etc.). Even if the model doesn't use them heavily, having them available lets the post-hoc analysis explore whether flagged anomalies cluster by time period.

"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### D.n Fixing "\<line_total_computed + finalise prod_offline_median_ratio\>"

> You can add more cells related to new features in this section

In [ ]:
for d in [training_df_eng, validation_df_eng, testing_df_eng]:
    d['line_total_computed'] = d['order_quantity'] * d['net_unit_price']

print("D.4 added: line_total_computed")
print(f"\nprod_offline_median_ratio nulls (online-only products):")
print(f"  train: {training_df_eng['prod_offline_median_ratio'].isnull().sum()}/{len(training_df_eng)}")
print(f"  val:   {validation_df_eng['prod_offline_median_ratio'].isnull().sum()}/{len(validation_df_eng)}")
print(f"  test:  {testing_df_eng['prod_offline_median_ratio'].isnull().sum()}/{len(testing_df_eng)}")

# Online-only products never traded offline -> impute 1.0 (they only trade at full retail)
for d in [training_df_eng, validation_df_eng, testing_df_eng]:
    d['prod_offline_median_ratio'] = d['prod_offline_median_ratio'].fillna(1.0)
print("After fillna with 1.0: 0 nulls everywhere")

print(f"\ntrain shape after engineering: {training_df_eng.shape}")
print(f"val   shape after engineering: {validation_df_eng.shape}")
print(f"test  shape after engineering: {testing_df_eng.shape}")


In [ ]:
engineered_cols = ['net_unit_price','price_vs_listprice','margin_pct',
                   'qty_vs_subcat_median','price_vs_subcat_median',
                   'order_year','order_month','order_dow',
                   'prod_offline_median_ratio','line_total_computed']
print(f"Final engineered features added in Section D ({len(engineered_cols)}):")
for c in engineered_cols:
    print(f"  - {c}")

In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_n_explanations = """
D.4 wraps up engineering with two final additions:

1. line_total_computed = order_quantity * net_unit_price: a redundant magnitude feature. Even though we have line_total already, having it expressed as a function of quantity and discounted price helps tree models split on a more interpretable axis.
2. prod_offline_median_ratio finalised: for products that never traded offline (6,814 train, 2,276 val, 2,268 test lines) we impute the value 1.0, representing 'this product only trades at full retail online'. This keeps the column dense and the model can still use it to distinguish online-only products from those with a wholesale rate.

After Section D the engineered datasets contain 10 new columns on top of the 22 cleaned columns, plus 5 label-related helper columns. The full feature set used by the model will be a subset of these in Section E.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

---
## E. Data Preparation for Modeling

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()
except Exception as e:
  print(e)

### E.1 Data Transformation <put_name_here>


In [ ]:
NUMERIC_FEATURES = [
    'order_quantity','unit_price','unit_price_discount','line_total',
    'list_price','standard_cost','net_unit_price','price_vs_listprice','margin_pct',
    'price_vs_subcat_median','qty_vs_subcat_median','line_total_computed',
    'prod_offline_median_ratio',
    'order_year','order_month','order_dow','offered_discount_pct'
]
CATEGORICAL_FEATURES = ['online_order_flag','category_name','offer_type','offer_category']
TARGET = 'is_anomaly'

print(f"Numeric features ({len(NUMERIC_FEATURES)}):")
for f in NUMERIC_FEATURES: print(f"  - {f}")
print(f"\nCategorical features ({len(CATEGORICAL_FEATURES)}):")
for f in CATEGORICAL_FEATURES: print(f"  - {f}")
print(f"\nTarget: {TARGET}")

In [ ]:
y_train = X_train[TARGET].reset_index(drop=True)
y_val   = X_val[TARGET].reset_index(drop=True)
y_test  = X_test[TARGET].reset_index(drop=True)

print(f"y_train: shape={y_train.shape}, anomalies={y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"y_val:   shape={y_val.shape}, anomalies={y_val.sum()} ({y_val.mean()*100:.2f}%)")
print(f"y_test:  shape={y_test.shape}, anomalies={y_test.sum()} ({y_test.mean()*100:.2f}%)")

In [ ]:
data_transformation_1_explanations = """
E.1 freezes the final feature set used by the model: 17 numeric columns + 4 categorical columns = 21 raw features before one-hot encoding.

What we KEEP:
- The original 4 raw transactional columns (order_quantity, unit_price, unit_price_discount, line_total).
- Product reference values (list_price, standard_cost, offered_discount_pct).
- 7 engineered numeric features (net_unit_price, price_vs_listprice, margin_pct, price_vs_subcat_median, qty_vs_subcat_median, line_total_computed, prod_offline_median_ratio).
- 3 calendar features (order_year, order_month, order_dow).
- 4 low-cardinality categoricals (online_order_flag, category_name, offer_type, offer_category).

What we DROP:
- Bookkeeping IDs: product_id, sales_order_id, sales_order_detail_id, customer_id, territory_id (only 4 values), product_subcategory_id, product_category_id, special_offer_id.
- High-cardinality and sparse: subcategory_name (35 levels - would balloon columns), class (54% null).
- Helper columns from the label-building process (subcat_q99_qty, subcat_median_qty, subcat_median_price) - these would leak label information.
- The label sub-components (label_unusual_online_discount etc.) - pulled into y, removed from X.
- order_date (replaced by year/month/dow).

The target column is_anomaly is extracted into y_train / y_val / y_test BEFORE we start transforming X, so we never accidentally scale or encode the label.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### E.2 Data Transformation <put_name_here>

In [ ]:
# <Student to fill this section>
X_train_sub = X_train[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
X_val_sub   = X_val[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
X_test_sub  = X_test[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()

null_summary = X_train_sub[NUMERIC_FEATURES].isnull().sum()
print("Numeric null counts before impute (train):")
print(null_summary[null_summary > 0])

medians = X_train_sub[NUMERIC_FEATURES].median()
X_train_sub[NUMERIC_FEATURES] = X_train_sub[NUMERIC_FEATURES].fillna(medians)
X_val_sub[NUMERIC_FEATURES]   = X_val_sub[NUMERIC_FEATURES].fillna(medians)
X_test_sub[NUMERIC_FEATURES]  = X_test_sub[NUMERIC_FEATURES].fillna(medians)
print(f"\nAfter impute (train): {X_train_sub[NUMERIC_FEATURES].isnull().sum().sum()} remaining")

for col in CATEGORICAL_FEATURES:
    X_train_sub[col] = X_train_sub[col].astype(str)
    X_val_sub[col]   = X_val_sub[col].astype(str)
    X_test_sub[col]  = X_test_sub[col].astype(str)

X_train_enc = pd.get_dummies(X_train_sub, columns=CATEGORICAL_FEATURES, drop_first=True)
X_val_enc   = pd.get_dummies(X_val_sub,   columns=CATEGORICAL_FEATURES, drop_first=True)
X_test_enc  = pd.get_dummies(X_test_sub,  columns=CATEGORICAL_FEATURES, drop_first=True)
X_val_enc  = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print(f"\nAfter one-hot encoding:")
print(f"  X_train_enc: {X_train_enc.shape}")
print(f"  X_val_enc:   {X_val_enc.shape}")
print(f"  X_test_enc:  {X_test_enc.shape}")
print(f"\nFinal feature columns ({len(X_train_enc.columns)}):")
for c in X_train_enc.columns:
    print(f"  - {c}")

In [ ]:
print(f"X_train_enc nulls: {X_train_enc.isnull().sum().sum()}")
print(f"X_val_enc nulls:   {X_val_enc.isnull().sum().sum()}")
print(f"X_test_enc nulls:  {X_test_enc.isnull().sum().sum()}")
print(f"\nAll columns identical across splits: {list(X_train_enc.columns) == list(X_val_enc.columns) == list(X_test_enc.columns)}")

In [ ]:
data_transformation_2_explanations = """
E.2 handles missing values and converts categoricals to a numeric form:

1. Null imputation (numeric): we apply train-set medians to fill any remaining numeric nulls. In this run there were no remaining nulls (the hierarchical imputation in B.3 plus the fillna(1.0) in D.4 covered everything), but the safety net is in place for re-runs.
2. Categorical encoding: one-hot encode the 4 low-cardinality categoricals with drop_first=True to avoid the dummy-variable trap. After encoding the final dimensionality is 27 features.
3. Column alignment: pandas.get_dummies can produce different columns if a categorical level only appears in train. We reindex val/test to match X_train_enc's columns, filling any missing column with 0. This guarantees identical feature dimensionality across the three datasets.

We keep all encoded category_name levels (Bikes, Clothing, Components, with Accessories as the dropped reference) because the model needs to know which category each line belongs to. Similarly we keep all five offer_type levels.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### E.3 Data Transformation <put_name_here>


In [ ]:
scaler = StandardScaler()
X_train_enc[NUMERIC_FEATURES] = scaler.fit_transform(X_train_enc[NUMERIC_FEATURES])
X_val_enc[NUMERIC_FEATURES]   = scaler.transform(X_val_enc[NUMERIC_FEATURES])
X_test_enc[NUMERIC_FEATURES]  = scaler.transform(X_test_enc[NUMERIC_FEATURES])

print("Numeric features now standardised (mean=0, std=1 on train).")
print("\nSample summary of scaled features (train, first 6):")
print(X_train_enc[NUMERIC_FEATURES[:6]].describe().round(3).T[['mean','std','min','max']])

In [ ]:
X_train = X_train_enc.reset_index(drop=True)
X_val   = X_val_enc.reset_index(drop=True)
X_test  = X_test_enc.reset_index(drop=True)

print(f"Final modeling artefacts:")
print(f"  X_train: {X_train.shape}, dtypes: {X_train.dtypes.value_counts().to_dict()}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}, y_val: {y_val.shape}, y_test: {y_test.shape}")

In [ ]:
data_transformation_3_explanations = """
E.3 standardises numeric features using StandardScaler (fit on train only).
Why scale: although the primary model (Isolation Forest) is scale-invariant, we will also train comparison models (One-Class SVM, Local Outlier Factor) that DO depend on Euclidean distances and require feature scaling for fair distance computation. Scaling everything once now ensures all candidate models train on the same input.

The fit on train / transform on val and test pattern is essential to avoid data leakage. The train mean and std are computed from training data only, then applied as constants to val and test.

Bool columns (one-hot indicators) are left unscaled - their values are already 0 or 1 and scikit-learn handles the mixed dtype automatically.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

### E.n Fixing "\<describe_issue_here\>"

> You can add more cells related to data preparation in this section

In [ ]:
y_train.to_csv(at.folder_path / 'y_train.csv', index=False)
y_val.to_csv(at.folder_path / 'y_val.csv', index=False)
y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
print("y_train.csv, y_val.csv, y_test.csv saved to assignment folder.")

for fname in ['y_train.csv','y_val.csv','y_test.csv']:
    loaded = pd.read_csv(at.folder_path / fname)
    print(f"  {fname}: shape={loaded.shape}, anomalies={loaded['is_anomaly'].sum()}")

In [ ]:
print("="*60)
print("PREPARATION COMPLETE")
print("="*60)
print(f"X_train: {X_train.shape}  y_train anomaly rate: {y_train.mean()*100:.2f}%")
print(f"X_val:   {X_val.shape}  y_val   anomaly rate: {y_val.mean()*100:.2f}%")
print(f"X_test:  {X_test.shape}  y_test  anomaly rate: {y_test.mean()*100:.2f}%")
print(f"\nTotal feature columns: {X_train.shape[1]}")
print(f"Numeric features (scaled): {len(NUMERIC_FEATURES)}")
print(f"One-hot encoded indicators: {X_train.shape[1] - len(NUMERIC_FEATURES)}")


In [ ]:
data_transformation_n_explanations = """
1. E.1 Selected 17 numeric + 4 categorical features (21 raw inputs), extracted target.
2. E.2 Imputed numeric nulls with train medians (0 remaining nulls in this run); one-hot encoded categoricals with drop_first; aligned val/test columns to train. Final dimensionality: 27.
3. E.3 Standardised all 17 numeric features using StandardScaler fit on train only.
4. E.4 Saved y_train.csv, y_val.csv, y_test.csv separately because the Baseline notebook's load block expects them and the Section F save block only handles X.

The final modeling artefacts:
- X_train (25,260 x 27), X_val (8,420 x 27), X_test (8,420 x 27): scaled numeric + one-hot categorical features. No nulls. All float / bool dtypes compatible with scikit-learn.
- y_train (25,260), y_val (8,420), y_test (8,420): binary 'is_anomaly' label at 5.58% positive rate in every split.

"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_n_explanations', value=data_transformation_n_explanations)

---
## F. Save Datasets

> Do not change this code

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
except Exception as e:
  print(e)